# Schedule Reliability Analysis

Analyze how much RPI shuttles deviate from their static schedule times.

This notebook:
1. Loads GPS data from CSV and the static schedule from `schedule.json`
2. Labels GPS points with stop information using vectorized haversine distance
3. Detects stop arrival events (first arrival at each stop per trip)
4. Matches vehicles to scheduled bus routes using the Hungarian algorithm
5. Computes signed deviation in minutes (positive = late, negative = early)
6. Produces summary statistics and visualizations

## Cell 1: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
from pathlib import Path
from datetime import datetime, timedelta
import pytz
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.optimize import linear_sum_assignment

# Add project root to Python path so we can import shared modules
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from shared.stops import Stops, haversine_vectorized

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

# Timezone
ET = pytz.timezone("America/New_York")

# --- Load GPS data ---
csv_path = Path("cache/shared/shubble-sample.csv")
df = pd.read_csv(csv_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["vehicle_id", "timestamp"]).reset_index(drop=True)

# --- Load schedule and routes ---
with open(project_root / "shared" / "schedule.json") as f:
    schedule_data = json.load(f)

with open(project_root / "shared" / "routes.json") as f:
    routes_data = json.load(f)

# --- Basic stats ---
print(f"Total GPS rows:    {len(df):,}")
print(f"Date range:        {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Unique vehicles:   {df['vehicle_id'].nunique()}")
print(f"Columns:           {list(df.columns)}")
print(f"\nSchedule day types: { {d: schedule_data[d] for d in ['MONDAY','TUESDAY','WEDNESDAY','THURSDAY','FRIDAY','SATURDAY','SUNDAY']} }")
print(f"Weekday schedules: {list(schedule_data['weekday'].keys())}")
df.head()

## Cell 2: Label GPS Points with Stop Info

Use vectorized haversine distance to check each GPS point against all known stops.
A point is "at a stop" if it is within 20m (0.020 km) of any stop coordinate.
We keep only on-route points (where `route_name` is not None).

In [ ]:
STOP_THRESHOLD_KM = 0.020  # 20 meters

# Build arrays of all stop coordinates, route names, and stop names from routes.json
all_stop_coords = []  # list of [lat, lon]
all_stop_route_names = []
all_stop_names = []

for route_name, route in routes_data.items():
    for stop_name in route.get("STOPS", []):
        coords = route[stop_name]["COORDINATES"]
        all_stop_coords.append(coords)
        all_stop_route_names.append(route_name)
        all_stop_names.append(stop_name)

stops_arr = np.array(all_stop_coords)  # shape (num_stops, 2)
print(f"Total stops across all routes: {len(stops_arr)}")

# Vectorized labeling: compute distance from each GPS point to every stop
# Process in chunks to avoid memory issues with 500K x num_stops matrix
gps_coords = df[["latitude", "longitude"]].values  # shape (N, 2)
num_points = len(gps_coords)
num_stops = len(stops_arr)

route_labels = np.full(num_points, None, dtype=object)
stop_labels = np.full(num_points, None, dtype=object)

CHUNK_SIZE = 50_000
for start in range(0, num_points, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, num_points)
    chunk = gps_coords[start:end]  # shape (chunk_size, 2)

    # Compute distances: chunk_size x num_stops
    # Broadcast: expand chunk to (chunk_size, 1, 2) and stops to (1, num_stops, 2)
    chunk_lat = np.radians(chunk[:, 0:1])   # (chunk, 1)
    chunk_lon = np.radians(chunk[:, 1:2])   # (chunk, 1)
    stops_lat = np.radians(stops_arr[:, 0]) # (num_stops,)
    stops_lon = np.radians(stops_arr[:, 1]) # (num_stops,)

    dlat = stops_lat[np.newaxis, :] - chunk_lat  # (chunk, num_stops)
    dlon = stops_lon[np.newaxis, :] - chunk_lon  # (chunk, num_stops)

    a = np.sin(dlat / 2) ** 2 + np.cos(chunk_lat) * np.cos(stops_lat[np.newaxis, :]) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distances = 6371.0 * c  # km, shape (chunk, num_stops)

    # Find closest stop for each point
    min_idx = np.argmin(distances, axis=1)
    min_dist = distances[np.arange(len(chunk)), min_idx]

    # Label points within threshold
    within = min_dist < STOP_THRESHOLD_KM
    for i in np.where(within)[0]:
        global_i = start + i
        si = min_idx[i]
        route_labels[global_i] = all_stop_route_names[si]
        stop_labels[global_i] = all_stop_names[si]

    if (end // CHUNK_SIZE) % 2 == 0 or end == num_points:
        print(f"  Processed {end:,}/{num_points:,} points...")

df["route_name"] = route_labels
df["stop_name"] = stop_labels

# Summary
at_stop_mask = df["route_name"].notna()
print(f"\nPoints at a stop:     {at_stop_mask.sum():,} ({at_stop_mask.mean()*100:.1f}%)")
print(f"Points NOT at a stop: {(~at_stop_mask).sum():,} ({(~at_stop_mask).mean()*100:.1f}%)")

# Filter to on-route points only
df_on_route = df[at_stop_mask].copy().reset_index(drop=True)
print(f"\nOn-route DataFrame shape: {df_on_route.shape}")
print(f"Routes found: {df_on_route['route_name'].value_counts().to_dict()}")
print(f"Stops found:  {df_on_route['stop_name'].value_counts().to_dict()}")

## Cell 3: Detect Stop Arrival Events

A vehicle "arrives" at a stop when it transitions from not being at that stop to being at that stop.
For each (vehicle_id, date, route_name, stop_name) combination, we record the earliest timestamp of each distinct visit.

In [ ]:
# Convert timestamps to Eastern Time for schedule comparison
df_on_route["timestamp_et"] = df_on_route["timestamp"].dt.tz_convert(ET)
df_on_route["date"] = df_on_route["timestamp_et"].dt.date

# Detect arrival events:
# A new arrival occurs when the current stop differs from the previous stop
# for the same vehicle, OR it is the first point for that vehicle on that date.
df_on_route = df_on_route.sort_values(["vehicle_id", "timestamp"]).reset_index(drop=True)

# Create a "previous stop" column per vehicle
df_on_route["prev_stop"] = df_on_route.groupby("vehicle_id")["stop_name"].shift(1)
df_on_route["prev_route"] = df_on_route.groupby("vehicle_id")["route_name"].shift(1)

# An arrival event is when stop_name != prev_stop OR route_name != prev_route
# (i.e., the vehicle just reached this stop)
is_new_arrival = (
    (df_on_route["stop_name"] != df_on_route["prev_stop"]) |
    (df_on_route["route_name"] != df_on_route["prev_route"]) |
    df_on_route["prev_stop"].isna()
)

arrivals = df_on_route[is_new_arrival][
    ["vehicle_id", "date", "route_name", "stop_name", "timestamp_et"]
].copy()
arrivals = arrivals.rename(columns={"timestamp_et": "arrival_time"})
arrivals = arrivals.reset_index(drop=True)

print(f"Total arrival events detected: {len(arrivals):,}")
print(f"Unique dates: {arrivals['date'].nunique()}")
print(f"Arrivals per route:\n{arrivals['route_name'].value_counts()}")
print(f"\nArrivals per stop (top 10):\n{arrivals['stop_name'].value_counts().head(10)}")
arrivals.head(10)

## Cell 4: Schedule Matching Per Day

For each date, determine the day type (weekday/saturday/sunday), get the scheduled bus routes,
and use the Hungarian algorithm to optimally match vehicles to bus schedules.

The cost matrix is based on how well each vehicle's actual STUDENT_UNION arrivals match the
scheduled departure times for each bus schedule. Cost = 1 - (matches / total_scheduled_stops).

In [ ]:
import calendar

# Map day-of-week to schedule.json day name
DOW_TO_DAY = {
    0: "MONDAY", 1: "TUESDAY", 2: "WEDNESDAY", 3: "THURSDAY",
    4: "FRIDAY", 5: "SATURDAY", 6: "SUNDAY"
}

# First stop for each route (the departure point referenced in the schedule)
FIRST_STOP = {
    "WEST": "STUDENT_UNION",
    "NORTH": "STUDENT_UNION",
}

def parse_schedule_time(time_str, date):
    """Parse a schedule time like '7:00 AM' into a timezone-aware datetime on the given date."""
    dt = datetime.strptime(time_str, "%I:%M %p")
    combined = datetime(date.year, date.month, date.day, dt.hour, dt.minute)
    # Handle midnight (12:00 AM) -- this means the next calendar day
    if dt.hour == 0 and dt.minute == 0:
        combined += timedelta(days=1)
    return ET.localize(combined)


def get_day_schedule(date):
    """Get the schedule entries for a given date. Returns dict of {schedule_name: [(time, route), ...]}."""
    dow = date.weekday()
    day_name = DOW_TO_DAY[dow]
    day_type = schedule_data.get(day_name)
    if day_type is None or day_type not in schedule_data:
        return None, day_type
    return schedule_data[day_type], day_type


def match_vehicles_to_schedules(date, arrivals_day):
    """
    Match vehicles to bus schedules for a given day using the Hungarian algorithm.

    Returns: dict of {vehicle_id: schedule_name} and the schedule dict.
    """
    day_schedule, day_type = get_day_schedule(date)
    if day_schedule is None:
        return {}, None, day_type

    # Get vehicles active on this day
    vehicle_ids = arrivals_day["vehicle_id"].unique()
    if len(vehicle_ids) == 0:
        return {}, day_schedule, day_type

    schedule_names = list(day_schedule.keys())

    # For each vehicle, get its STUDENT_UNION arrival times with route info
    vehicle_su_arrivals = {}
    for vid in vehicle_ids:
        vid_arrivals = arrivals_day[arrivals_day["vehicle_id"] == vid]
        # Filter to first-stop arrivals (STUDENT_UNION for both routes)
        su_arrivals = vid_arrivals[
            vid_arrivals["stop_name"].isin(FIRST_STOP.values())
        ].copy()
        if len(su_arrivals) > 0:
            # Store as list of (arrival_time, route_name)
            vehicle_su_arrivals[vid] = list(
                zip(su_arrivals["arrival_time"], su_arrivals["route_name"])
            )

    active_vehicles = list(vehicle_su_arrivals.keys())
    if len(active_vehicles) == 0:
        return {}, day_schedule, day_type

    # Build cost matrix: vehicles x schedules
    # Cost = 1 - (fraction of scheduled departures matched by actual arrivals)
    # A "match" means the vehicle arrived at the correct route's first stop within 15 min of scheduled time
    MATCH_WINDOW_MIN = 15

    cost_matrix = np.ones((len(active_vehicles), len(schedule_names)))

    for v_idx, vid in enumerate(active_vehicles):
        v_arrivals = vehicle_su_arrivals[vid]
        for s_idx, sched_name in enumerate(schedule_names):
            sched_entries = day_schedule[sched_name]
            total = len(sched_entries)
            if total == 0:
                continue

            matches = 0
            for time_str, route_name in sched_entries:
                sched_time = parse_schedule_time(time_str, date)
                first_stop = FIRST_STOP.get(route_name, "STUDENT_UNION")

                # Check if vehicle arrived at this route's first stop near this time
                for arr_time, arr_route in v_arrivals:
                    if arr_route == route_name:
                        diff_min = abs((arr_time - sched_time).total_seconds()) / 60
                        if diff_min <= MATCH_WINDOW_MIN:
                            matches += 1
                            break

            cost_matrix[v_idx, s_idx] = 1.0 - (matches / total)

    # Solve assignment problem
    # If more schedules than vehicles, pad; if more vehicles than schedules, that is fine
    # linear_sum_assignment handles rectangular matrices
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    assignments = {}
    for r, c in zip(row_ind, col_ind):
        vid = active_vehicles[r]
        sched_name = schedule_names[c]
        match_score = 1.0 - cost_matrix[r, c]
        # Only assign if the vehicle matched at least some stops
        if match_score > 0.05:  # at least 5% of stops matched
            assignments[vid] = sched_name

    return assignments, day_schedule, day_type


# Run matching for each date
all_assignments = {}  # date -> {vehicle_id: schedule_name}
all_day_schedules = {}  # date -> schedule dict

unique_dates = sorted(arrivals["date"].unique())
print(f"Processing {len(unique_dates)} unique dates...\n")

for date in unique_dates:
    arrivals_day = arrivals[arrivals["date"] == date]
    assignments, day_schedule, day_type = match_vehicles_to_schedules(date, arrivals_day)
    all_assignments[date] = assignments
    all_day_schedules[date] = day_schedule

    n_assigned = len(assignments)
    n_vehicles = arrivals_day["vehicle_id"].nunique()
    print(f"  {date} ({day_type}): {n_assigned}/{n_vehicles} vehicles matched to schedules"
          + (f" -> {assignments}" if n_assigned > 0 else ""))

total_assigned = sum(len(v) for v in all_assignments.values())
print(f"\nTotal vehicle-day assignments: {total_assigned}")

## Cell 5: Compute Deviations

For each matched (vehicle, schedule) pair, compare actual STUDENT_UNION arrival times to scheduled departure times.
Deviation = actual_arrival - scheduled_time (positive = late, negative = early).

In [ ]:
# For each (date, vehicle_id, schedule_name) assignment, compare actual arrivals to schedule
deviation_records = []

for date, assignments in all_assignments.items():
    day_schedule = all_day_schedules[date]
    if day_schedule is None:
        continue

    arrivals_day = arrivals[arrivals["date"] == date]

    for vid, sched_name in assignments.items():
        sched_entries = day_schedule[sched_name]
        vid_arrivals = arrivals_day[
            (arrivals_day["vehicle_id"] == vid) &
            (arrivals_day["stop_name"].isin(FIRST_STOP.values()))
        ]

        # Build list of (arrival_time, route_name) for this vehicle
        v_arr_list = list(zip(vid_arrivals["arrival_time"], vid_arrivals["route_name"]))

        for time_str, route_name in sched_entries:
            sched_time = parse_schedule_time(time_str, date)
            first_stop = FIRST_STOP.get(route_name, "STUDENT_UNION")

            # Find the closest actual arrival at this route's first stop
            best_arrival = None
            best_diff = float("inf")

            for arr_time, arr_route in v_arr_list:
                if arr_route == route_name:
                    diff_sec = (arr_time - sched_time).total_seconds()
                    # Only consider arrivals within a reasonable window (30 min)
                    if abs(diff_sec) < 30 * 60 and abs(diff_sec) < abs(best_diff * 60):
                        best_diff = diff_sec / 60.0
                        best_arrival = arr_time

            if best_arrival is not None:
                deviation_records.append({
                    "date": date,
                    "vehicle_id": vid,
                    "schedule_name": sched_name,
                    "route_name": route_name,
                    "scheduled_time": sched_time,
                    "actual_arrival": best_arrival,
                    "deviation_minutes": best_diff,
                })

deviations = pd.DataFrame(deviation_records)
print(f"Total deviation measurements: {len(deviations):,}")

if len(deviations) > 0:
    print(f"Date range: {deviations['date'].min()} to {deviations['date'].max()}")
    print(f"Unique vehicles: {deviations['vehicle_id'].nunique()}")
    print(f"Unique schedules: {deviations['schedule_name'].nunique()}")
    print(f"\nSample deviations:")
    display_cols = ["date", "vehicle_id", "schedule_name", "route_name",
                    "scheduled_time", "actual_arrival", "deviation_minutes"]
    print(deviations[display_cols].head(15).to_string(index=False))
else:
    print("No deviation measurements found. Check that GPS data overlaps with schedule times.")

## Cell 6: Summary Statistics

In [ ]:
if len(deviations) > 0:
    dev = deviations["deviation_minutes"]

    print("=" * 60)
    print("OVERALL DEVIATION STATISTICS")
    print("=" * 60)
    print(f"  Mean deviation:   {dev.mean():+.2f} min")
    print(f"  Median deviation: {dev.median():+.2f} min")
    print(f"  Std deviation:    {dev.std():.2f} min")
    print(f"  Min deviation:    {dev.min():+.2f} min")
    print(f"  Max deviation:    {dev.max():+.2f} min")

    print(f"\n  Within +/- 2 min:  {(dev.abs() <= 2).mean()*100:.1f}%")
    print(f"  Within +/- 5 min:  {(dev.abs() <= 5).mean()*100:.1f}%")
    print(f"  Within +/- 10 min: {(dev.abs() <= 10).mean()*100:.1f}%")
    print(f"  Late (>0 min):     {(dev > 0).mean()*100:.1f}%")
    print(f"  Early (<0 min):    {(dev < 0).mean()*100:.1f}%")
    print(f"  On time (0 min):   {(dev == 0).mean()*100:.1f}%")

    # By route
    print("\n" + "=" * 60)
    print("DEVIATION BY ROUTE")
    print("=" * 60)
    route_stats = deviations.groupby("route_name")["deviation_minutes"].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
    print(route_stats.round(2).to_string())

    # By schedule name
    print("\n" + "=" * 60)
    print("DEVIATION BY SCHEDULE")
    print("=" * 60)
    sched_stats = deviations.groupby("schedule_name")["deviation_minutes"].agg(
        ["count", "mean", "median", "std"]
    )
    print(sched_stats.round(2).to_string())

    # By time of day
    print("\n" + "=" * 60)
    print("DEVIATION BY TIME OF DAY")
    print("=" * 60)
    deviations["hour"] = deviations["scheduled_time"].dt.hour
    deviations["time_of_day"] = pd.cut(
        deviations["hour"],
        bins=[0, 9, 12, 17, 24],
        labels=["Early Morning (0-9)", "Morning (9-12)", "Afternoon (12-17)", "Evening (17-24)"],
        right=False
    )
    tod_stats = deviations.groupby("time_of_day", observed=True)["deviation_minutes"].agg(
        ["count", "mean", "median", "std"]
    )
    print(tod_stats.round(2).to_string())

    # Worst offenders: time slots with highest absolute deviation
    print("\n" + "=" * 60)
    print("TOP 10 WORST DEVIATIONS (largest absolute deviation)")
    print("=" * 60)
    worst = deviations.nlargest(10, "deviation_minutes")[
        ["date", "schedule_name", "route_name", "scheduled_time", "deviation_minutes"]
    ]
    print(worst.to_string(index=False))
else:
    print("No deviations to analyze.")

## Cell 7: Visualizations

In [ ]:
if len(deviations) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    # --- 1. Histogram of all deviations ---
    ax = axes[0, 0]
    ax.hist(deviations["deviation_minutes"], bins=50, edgecolor="black", alpha=0.7, color="steelblue")
    ax.axvline(0, color="red", linestyle="--", linewidth=1.5, label="On time")
    ax.axvline(deviations["deviation_minutes"].mean(), color="orange", linestyle="--",
               linewidth=1.5, label=f"Mean ({deviations['deviation_minutes'].mean():.1f} min)")
    ax.set_xlabel("Deviation (minutes)")
    ax.set_ylabel("Frequency")
    ax.set_title("Distribution of Schedule Deviations")
    ax.legend()

    # --- 2. Box plot by route ---
    ax = axes[0, 1]
    route_groups = [
        deviations[deviations["route_name"] == r]["deviation_minutes"].dropna()
        for r in sorted(deviations["route_name"].unique())
    ]
    route_labels_sorted = sorted(deviations["route_name"].unique())
    bp = ax.boxplot(route_groups, labels=route_labels_sorted, patch_artist=True)
    colors = ["#4C72B0", "#DD8452"]
    for patch, color in zip(bp["boxes"], colors[:len(route_labels_sorted)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("Route")
    ax.set_ylabel("Deviation (minutes)")
    ax.set_title("Deviation by Route (NORTH vs WEST)")

    # --- 3. Box plot by time of day ---
    ax = axes[0, 2]
    tod_groups = []
    tod_labels = []
    for tod in deviations["time_of_day"].cat.categories:
        subset = deviations[deviations["time_of_day"] == tod]["deviation_minutes"].dropna()
        if len(subset) > 0:
            tod_groups.append(subset)
            tod_labels.append(str(tod).replace(" ", "\n"))
    if tod_groups:
        bp2 = ax.boxplot(tod_groups, labels=tod_labels, patch_artist=True)
        tod_colors = ["#55A868", "#C44E52", "#8172B2", "#CCB974"]
        for patch, color in zip(bp2["boxes"], tod_colors[:len(tod_groups)]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("Time of Day")
    ax.set_ylabel("Deviation (minutes)")
    ax.set_title("Deviation by Time of Day")

    # --- 4. Scatter: scheduled time vs deviation, colored by route ---
    ax = axes[1, 0]
    for route_name, color in zip(sorted(deviations["route_name"].unique()), ["#4C72B0", "#DD8452"]):
        subset = deviations[deviations["route_name"] == route_name]
        ax.scatter(
            subset["scheduled_time"].dt.hour + subset["scheduled_time"].dt.minute / 60,
            subset["deviation_minutes"],
            alpha=0.4, s=20, label=route_name, color=color
        )
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("Scheduled Hour of Day")
    ax.set_ylabel("Deviation (minutes)")
    ax.set_title("Scheduled Time vs Deviation")
    ax.legend()

    # --- 5. Per-day average deviation trend ---
    ax = axes[1, 1]
    daily_avg = deviations.groupby("date")["deviation_minutes"].agg(["mean", "std", "count"])
    daily_avg = daily_avg[daily_avg["count"] >= 3]  # at least 3 measurements
    if len(daily_avg) > 0:
        dates_plot = pd.to_datetime(daily_avg.index)
        ax.plot(dates_plot, daily_avg["mean"], marker="o", markersize=4, color="steelblue", linewidth=1.5)
        ax.fill_between(
            dates_plot,
            daily_avg["mean"] - daily_avg["std"],
            daily_avg["mean"] + daily_avg["std"],
            alpha=0.2, color="steelblue"
        )
        ax.axhline(0, color="red", linestyle="--", linewidth=1)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax.set_xlabel("Date")
    ax.set_ylabel("Mean Deviation (minutes)")
    ax.set_title("Daily Average Deviation Trend (+/- 1 std)")

    # --- 6. Heatmap: schedule name vs date, colored by mean deviation ---
    ax = axes[1, 2]
    if deviations["schedule_name"].nunique() > 1 and deviations["date"].nunique() > 1:
        pivot = deviations.pivot_table(
            values="deviation_minutes", index="schedule_name", columns="date", aggfunc="mean"
        )
        # Only show if there is enough data
        if pivot.notna().sum().sum() > 5:
            sns.heatmap(pivot, ax=ax, cmap="RdYlGn_r", center=0, annot=False,
                        cbar_kws={"label": "Mean Deviation (min)"})
            ax.set_title("Mean Deviation: Schedule x Date")
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
        else:
            ax.text(0.5, 0.5, "Insufficient data\nfor heatmap", ha="center", va="center",
                    transform=ax.transAxes, fontsize=14)
            ax.set_title("Mean Deviation: Schedule x Date")
    else:
        ax.text(0.5, 0.5, "Insufficient data\nfor heatmap", ha="center", va="center",
                transform=ax.transAxes, fontsize=14)
        ax.set_title("Mean Deviation: Schedule x Date")

    plt.tight_layout()
    plt.savefig("schedule_reliability_plots.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plots saved to schedule_reliability_plots.png")
else:
    print("No deviations to visualize.")

## Cell 8: Conclusions

### Key Findings

**Methodology:**
- GPS data was labeled with stop proximity (20m threshold) using vectorized haversine distance
- Stop arrival events were detected by identifying transitions between stops
- Vehicles were matched to bus schedules using the Hungarian algorithm, optimizing for the best overall assignment based on how well each vehicle's STUDENT_UNION arrivals aligned with scheduled departure times
- Signed deviations were computed as `actual_arrival - scheduled_time` (positive = late, negative = early)

**What to look for in the results above:**
- **Mean deviation** indicates overall tendency (positive = system runs late on average)
- **Percentage within +/- 5 min** is the key reliability metric -- transit agencies typically target 75%+
- **Route comparison** (NORTH vs WEST) reveals if one route is systematically less reliable
- **Time-of-day patterns** show if rush hours cause more delays
- **Daily trends** expose whether reliability degrades over time or varies by day of week
- **The heatmap** reveals which specific bus schedules are most problematic on which days

**Limitations:**
- GPS sampling rate affects arrival time precision (if GPS pings every 10s, arrival times have ~10s uncertainty)
- The 20m stop proximity threshold may miss arrivals if GPS accuracy is poor
- Schedule matching via Hungarian algorithm assumes each vehicle runs exactly one schedule per day
- Midnight crossover (12:00 AM entries) are handled as next-day but edge cases may exist
- The schedule only specifies departure from the first stop, so mid-route timing is not analyzed